In [1]:
import duckdb
import pandas as pd

# Conecta ao CSV direto via DuckDB (sem banco de dados)
con = duckdb.connect()

# Registra a base consolidada como tabela virtual
df = pd.read_csv('../02_dados_tratados/base_consolidada.csv', sep=';', encoding='utf-8-sig')
con.register('mercado_financeiro', df)

# Testa a conexão
resultado = con.execute("SELECT COUNT(*) as total_linhas FROM mercado_financeiro").df()
print("✅ DuckDB conectado!")
print(f"   Total de linhas na tabela: {resultado['total_linhas'][0]}")
print(f"   Tabela: mercado_financeiro")



✅ DuckDB conectado!
   Total de linhas na tabela: 78
   Tabela: mercado_financeiro


In [2]:
# Query 1 — Ranking de ROE anualizado no trimestre mais recente
query1 = """
SELECT
    Banco,
    Categoria,
    ROUND(ROE_anualizado * 100, 2) AS roe_pct,
    ROUND("Lucro Líquido" / 1000000, 1) AS lucro_mi,
    ROUND("Patrimônio Líquido" / 1000000, 1) AS patrimonio_mi
FROM mercado_financeiro
WHERE Trimestre = '2025T4'
ORDER BY ROE_anualizado DESC
"""

df_q1 = con.execute(query1).df()
print("📊 Query 1 — Ranking de ROE T4 2025:")
print(df_q1.to_string(index=False))

📊 Query 1 — Ranking de ROE T4 2025:
                    Banco         Categoria  roe_pct  lucro_mi  patrimonio_mi
                   Nubank           Digital   176.53       5.8           13.1
                  C6 Bank           Digital    92.04       1.4            6.1
Mercado Pago (Financeiro) Fintech Pagamento    63.23       0.4            2.3
                     Itaú       Tradicional    46.87      24.0          205.0
                Santander       Tradicional    34.45       8.3           96.4
                    Inter           Digital    32.18       0.6            7.3
                 Bradesco       Tradicional    28.60      13.1          182.7
          Banco do Brasil       Tradicional    18.23       8.5          185.7
                  PagBank Fintech Pagamento     7.69       0.0            1.4
 Mercado Pago (Pagamento) Fintech Pagamento     1.85       0.0            3.2
                   PicPay Fintech Pagamento   -59.07      -0.2            1.1


In [3]:
# Query 2 — Crescimento Year-over-Year de Ativo Total
# Compara T4 2025 com T4 2024 para cada banco
query2 = """
SELECT
    t1.Banco,
    t1.Categoria,
    ROUND(t1."Ativo Total" / 1000000000, 2) AS ativo_2025T4_bi,
    ROUND(t2."Ativo Total" / 1000000000, 2) AS ativo_2024T4_bi,
    ROUND(((t1."Ativo Total" - t2."Ativo Total") / t2."Ativo Total") * 100, 1) AS crescimento_yoy_pct
FROM mercado_financeiro t1
JOIN mercado_financeiro t2
    ON t1.Banco = t2.Banco
WHERE t1.Trimestre = '2025T4'
  AND t2.Trimestre = '2024T4'
ORDER BY crescimento_yoy_pct DESC
"""

df_q2 = con.execute(query2).df()
print("📊 Query 2 — Crescimento YoY de Ativo Total (2024T4 vs 2025T4):")
print(df_q2.to_string(index=False))

📊 Query 2 — Crescimento YoY de Ativo Total (2024T4 vs 2025T4):
                   Banco         Categoria  ativo_2025T4_bi  ativo_2024T4_bi  crescimento_yoy_pct
                  PicPay Fintech Pagamento             0.04             0.03                 49.7
Mercado Pago (Pagamento) Fintech Pagamento             0.09             0.06                 46.8
                  Nubank           Digital             0.25             0.18                 43.9
                   Inter           Digital             0.10             0.07                 28.8
                 C6 Bank           Digital             0.15             0.12                 27.5
                 PagBank Fintech Pagamento             0.05             0.04                 11.5
                Bradesco       Tradicional             1.94             1.74                 11.2
         Banco do Brasil       Tradicional             2.45             2.41                  1.8
                    Itaú       Tradicional             

In [4]:
# Query 3 — ROE médio nos 8 trimestres (visão histórica de eficiência)
query3 = """
SELECT
    Banco,
    Categoria,
    COUNT(Trimestre) AS trimestres_analisados,
    ROUND(AVG(ROE_anualizado) * 100, 2) AS roe_medio_pct,
    ROUND(MIN(ROE_anualizado) * 100, 2) AS roe_minimo_pct,
    ROUND(MAX(ROE_anualizado) * 100, 2) AS roe_maximo_pct
FROM mercado_financeiro
GROUP BY Banco, Categoria
ORDER BY roe_medio_pct DESC
"""

df_q3 = con.execute(query3).df()
print("📊 Query 3 — ROE medio historico (todos os trimestres):")
print(df_q3.to_string(index=False))

📊 Query 3 — ROE medio historico (todos os trimestres):
                    Banco         Categoria  trimestres_analisados  roe_medio_pct  roe_minimo_pct  roe_maximo_pct
                   Nubank           Digital                      8         120.69           68.24          182.97
                  C6 Bank           Digital                      8          75.97           45.12          111.20
Mercado Pago (Financeiro) Fintech Pagamento                      1          63.23           63.23           63.23
 Mercado Pago (Pagamento) Fintech Pagamento                      8          42.87            1.85           99.15
                     Itaú       Tradicional                      8          31.18           19.09           46.87
                Santander       Tradicional                      8          23.74           14.00           34.45
          Banco do Brasil       Tradicional                      8          23.41            6.92           42.12
                 Bradesco       T

In [5]:
# Query 4 — Evolução da Carteira de Crédito por categoria
query4 = """
SELECT
    Trimestre,
    Categoria,
    ROUND(SUM("Carteira de Crédito") / 1000000000, 2) AS carteira_bi
FROM mercado_financeiro
GROUP BY Trimestre, Categoria
ORDER BY Trimestre, Categoria
"""

df_q4 = con.execute(query4).df()
print("📊 Query 4 — Evolução da Carteira de Crédito por categoria (R$ bilhões):")
print(df_q4.to_string(index=False))

📊 Query 4 — Evolução da Carteira de Crédito por categoria (R$ bilhões):
Trimestre         Categoria  carteira_bi
   2024T1           Digital          NaN
   2024T1 Fintech Pagamento          NaN
   2024T1       Tradicional          NaN
   2024T2           Digital          NaN
   2024T2 Fintech Pagamento          NaN
   2024T2       Tradicional          NaN
   2024T3           Digital          NaN
   2024T3 Fintech Pagamento          NaN
   2024T3       Tradicional          NaN
   2024T4           Digital          NaN
   2024T4 Fintech Pagamento          NaN
   2024T4       Tradicional          NaN
   2025T1           Digital         0.15
   2025T1 Fintech Pagamento         0.06
   2025T1       Tradicional         3.56
   2025T2           Digital         0.17
   2025T2 Fintech Pagamento         0.07
   2025T2       Tradicional         3.58
   2025T3           Digital         0.18
   2025T3 Fintech Pagamento         0.08
   2025T3       Tradicional         3.59
   2025T4           Digita

In [6]:
# Query 5 — Ranking geral: ROE médio + crescimento de ativos combinados
query5 = """
WITH ativo_inicial AS (
    SELECT Banco, "Ativo Total" AS ativo_2024T1
    FROM mercado_financeiro
    WHERE Trimestre = '2024T1'
),
ativo_final AS (
    SELECT Banco, "Ativo Total" AS ativo_2025T4
    FROM mercado_financeiro
    WHERE Trimestre = '2025T4'
),
roe_medio AS (
    SELECT Banco, ROUND(AVG(ROE_anualizado) * 100, 1) AS roe_medio_pct
    FROM mercado_financeiro
    GROUP BY Banco
)
SELECT
    r.Banco,
    r.roe_medio_pct,
    ROUND(((af.ativo_2025T4 - ai.ativo_2024T1) / ai.ativo_2024T1) * 100, 1) AS crescimento_ativo_pct
FROM roe_medio r
JOIN ativo_inicial ai ON r.Banco = ai.Banco
JOIN ativo_final af ON r.Banco = af.Banco
ORDER BY roe_medio_pct DESC
"""

df_q5 = con.execute(query5).df()
print("📊 Query 5 — Ranking geral: ROE medio + Crescimento de Ativos:")
print(df_q5.to_string(index=False))# Query 5 — Ranking geral: ROE médio + crescimento de ativos combinados
query5 = """
WITH ativo_inicial AS (
    SELECT Banco, "Ativo Total" AS ativo_2024T1
    FROM mercado_financeiro
    WHERE Trimestre = '2024T1'
),
ativo_final AS (
    SELECT Banco, "Ativo Total" AS ativo_2025T4
    FROM mercado_financeiro
    WHERE Trimestre = '2025T4'
),
roe_medio AS (
    SELECT Banco, ROUND(AVG(ROE_anualizado) * 100, 1) AS roe_medio_pct
    FROM mercado_financeiro
    GROUP BY Banco
)
SELECT
    r.Banco,
    r.roe_medio_pct,
    ROUND(((af.ativo_2025T4 - ai.ativo_2024T1) / ai.ativo_2024T1) * 100, 1) AS crescimento_ativo_pct
FROM roe_medio r
JOIN ativo_inicial ai ON r.Banco = ai.Banco
JOIN ativo_final af ON r.Banco = af.Banco
ORDER BY roe_medio_pct DESC
"""

df_q5 = con.execute(query5).df()
print("📊 Query 5 — Ranking geral: ROE medio + Crescimento de Ativos:")
print(df_q5.to_string(index=False))

📊 Query 5 — Ranking geral: ROE medio + Crescimento de Ativos:
                   Banco  roe_medio_pct  crescimento_ativo_pct
                  Nubank          120.7                   98.1
                 C6 Bank           76.0                   93.3
Mercado Pago (Pagamento)           42.9                  140.7
                    Itaú           31.2                    8.4
               Santander           23.7                    7.6
         Banco do Brasil           23.4                    7.5
                Bradesco           19.0                   16.9
                   Inter           15.9                   57.1
                 PagBank            8.3                   42.0
📊 Query 5 — Ranking geral: ROE medio + Crescimento de Ativos:
                   Banco  roe_medio_pct  crescimento_ativo_pct
                  Nubank          120.7                   98.1
                 C6 Bank           76.0                   93.3
Mercado Pago (Pagamento)           42.9                  

In [7]:
# DOCUMENTAÇÃO DE LIMITAÇÃO — Carteira de Crédito 2024
# =====================================================
# Os trimestres de 2024 (2024T1 a 2024T4) retornam NaN para a coluna
# "Carteira de Crédito". Investigação indica que os CSVs da seção
# "Dados de 2000 a 2024" do IF.data utilizam nomenclatura ou estrutura
# de coluna diferente da seção "Dados a partir de 2025".
#
# DECISÃO ANALÍTICA:
# A análise de Carteira de Crédito será limitada aos 4 trimestres de 2025,
# onde os dados estão completos e consistentes.
#
# IMPACTO:
# - Queries de ROE, Ativo Total e Lucro Líquido: NÃO afetadas
# - Queries de Carteira de Crédito: limitadas a 2025T1-2025T4
#
# SUGESTÃO PARA VERSÃO FUTURA:
# Investigar nomes alternativos da coluna nos CSVs de 2024 e realizar
# mapeamento adicional no script de consolidação.

print("✅ Limitação documentada no notebook.")
print("   Carteira de Crédito: dados disponíveis apenas para 2025T1-2025T4")
print("   Demais métricas (ROE, Ativo Total, Lucro): série completa de 8 trimestres")

✅ Limitação documentada no notebook.
   Carteira de Crédito: dados disponíveis apenas para 2025T1-2025T4
   Demais métricas (ROE, Ativo Total, Lucro): série completa de 8 trimestres


In [8]:
# Cria o arquivo docs/metodologia.md com todas as decisões analíticas
metodologia = """# Metodologia e Decisões Analíticas

## Fonte de dados
- **IF.data / Banco Central do Brasil**
- Seção "Dados a partir de 2025": trimestres 2025T1 a 2025T4
- Seção "Dados de 2000 a 2024": trimestres 2024T1 a 2024T4
- Tipo de instituição: Conglomerados Financeiros e Instituições Independentes
- Relatório: Resumo

## Decisões analíticas

### 1. Banco Inter — múltiplas razões sociais
O Banco Inter aparece como `INTERMEDIUM` nos trimestres mais antigos
e como `INTER` nos mais recentes, refletindo a mudança de razão social
ocorrida em 2018. Ambas as entradas foram consolidadas sob o nome
comercial `Inter` para permitir análise temporal contínua.

### 2. Mercado Pago — duas entidades regulatórias
O Mercado Pago opera com duas entidades distintas no IF.data:
- `MERCADO PAGO INSTITUIÇÃO DE PAGAMENTO LTDA.` (entidade de pagamento)
- `MERCADO PAGO IP - FINANCEIRO` (entidade financeira, criada em 2025T4)

Decisão: manter as duas entidades separadas para preservar a riqueza
analítica e refletir a expansão regulatória da empresa.

### 3. ROE anualizado
O Lucro Líquido reportado no IF.data é trimestral.
Fórmula utilizada: `ROE_anualizado = (Lucro Líquido / Patrimônio Líquido) × 4`

### 4. Valores monetários
Os CSVs do IF.data utilizam ponto como separador de milhar (formato BR).
Conversão realizada via `str.replace('.', '')` antes do cast para float.
Todos os valores estão em R$ mil conforme padrão do IF.data.

## Limitações conhecidas

### Carteira de Crédito — dados incompletos em 2024
Os trimestres de 2024 (2024T1 a 2024T4) retornam NaN para a coluna
"Carteira de Crédito". A causa provável é divergência de nomenclatura
entre as duas seções do IF.data (2024 vs 2025+).

**Impacto:** análises de Carteira de Crédito limitadas a 2025T1-2025T4.
**Métricas não afetadas:** ROE, Ativo Total, Lucro Líquido, Patrimônio Líquido.

**Sugestão para versão futura:** mapear nomes alternativos da coluna
nos CSVs de 2024 e ajustar o script de consolidação.

## Período analisado
8 trimestres: 2024T1 a 2025T4 (2 anos completos)

## Instituições analisadas
| Banco | Categoria | Trimestres disponíveis |
|---|---|---|
| Itaú | Tradicional | 8 |
| Bradesco | Tradicional | 8 |
| Banco do Brasil | Tradicional | 8 |
| Santander | Tradicional | 8 |
| Nubank | Digital | 8 |
| C6 Bank | Digital | 8 |
| Inter | Digital | 8 |
| PagBank | Fintech Pagamento | 8 |
| Mercado Pago (Pagamento) | Fintech Pagamento | 8 |
| Mercado Pago (Financeiro) | Fintech Pagamento | 1 |
| PicPay | Fintech Pagamento | 5 |
"""

# Salva o arquivo
with open('../docs/metodologia.md', 'w', encoding='utf-8') as f:
    f.write(metodologia)

print("✅ docs/metodologia.md criado com sucesso!")
print("   Decisões analíticas documentadas: 4")
print("   Limitações documentadas: 1")

✅ docs/metodologia.md criado com sucesso!
   Decisões analíticas documentadas: 4
   Limitações documentadas: 1
